# 02 — The power-balance ledger and the Hall decomposition

The core claim of the comparator is an *exact* energy bookkeeping identity
(Drela 2009): `PK_pod − PK_BLI = ΔΦ_jet + ΔΦ_wake`. It is asserted at every
evaluation and enforced to 1e-10 relative in CI, so a broken comparison fails
loudly instead of biasing silently. This notebook shows the identity holding
across the design space, splits the benefit by mechanism (Hall et al. 2017),
and converts subsystem PSC to net PSC through the turboelectric chain.


In [ ]:
# resolve the repo root so the notebook runs from notebooks/ or the repo root
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "studies"))
import plotstyle

plotstyle.apply()
import matplotlib.pyplot as plt
import numpy as np

from blipb import BLIComparator
from blipb.powerbalance import hall2017

comp = BLIComparator()
res = comp.run_design(f_phi=0.5, fpr=1.25)
print(f"PK_pod - PK_BLI = {(res.pk_pod - res.pk_bli)/1e3:8.2f} kW")
print(f"dPhi_jet        = {res.delta_phi_jet/1e3:8.2f} kW")
print(f"dPhi_wake       = {res.delta_phi_wake/1e3:8.2f} kW")
print(f"ledger residual = {res.ledger_residual:.2e} (relative)")


In [ ]:
# The identity is exact everywhere, not just at the baseline
worst = 0.0
for f_phi in np.linspace(0.05, 0.95, 10):
    for fpr in np.linspace(1.2, 1.5, 7):
        r = comp.run_design(f_phi=f_phi, fpr=fpr)
        worst = max(worst, abs(r.ledger_residual))
print(f"max |ledger residual| over a 10 x 7 (f_phi, FPR) grid: {worst:.2e}")


In [ ]:
# Hall-2017 mechanism split: how much of the saving is avoided wake mixing
# vs reduced jet loss, as ingestion grows
fs = np.linspace(0.05, 0.95, 40)
dec = [hall2017.decompose(comp.run_design(f_phi=f, fpr=1.25)) for f in fs]

fig, ax = plt.subplots(figsize=(6, 3))
ax.stackplot(fs, [d.jet for d in dec], [d.wake for d in dec],
             labels=["jet mixing", "ingested wake"], alpha=0.8)
ax.plot(fs, [d.total for d in dec], "k--", lw=1, label="total PSC")
ax.set_xlabel(r"$f_\Phi$"); ax.set_ylabel("fraction of $P_{K,pod}$")
ax.legend(); ax.set_title("PSC by dissipation mechanism");

eff = hall2017.effective_fill_factor(comp.run_design(f_phi=0.5, fpr=1.25))
print(f"effective fill factor at the baseline point: {eff:.2f}")


In [ ]:
# Subsystem -> net: the turboelectric chain dilutes the benefit
fig, ax = plt.subplots(figsize=(6, 3))
runs = [comp.run_design(f_phi=f, fpr=1.25) for f in fs]
ax.plot(fs, [r.psc * 100 for r in runs], "k--", label="subsystem PSC")
for eta_elec in (0.85, 0.92, 0.98):
    net = [comp.net_psc(r, phi=0.28, eta_elec=eta_elec) * 100 for r in runs]
    ax.plot(fs, net, label=rf"net, $\eta_{{elec}}$ = {eta_elec}")
ax.set_xlabel(r"$f_\Phi$"); ax.set_ylabel("PSC [%]"); ax.legend()
ax.set_title(r"net PSC through the turboelectric chain ($\phi$ = 0.28)");


The strong sensitivity to `eta_elec` visible here is exactly what the Sobol'
analysis quantifies globally (notebook 05, Fig. 6): once ingestion fraction
passes ~0.5, the transmission chain — not the aerodynamics — controls the
variance of the net benefit.
